# RI #1: Building and Inspecting a Bounded Retrieval Agent

The single-pass baseline retrieves and answers along a fixed path. An agent
can choose tools and ask for more evidence. This notebook introduces the shared
starter pattern while keeping the task boundary, source provenance, and
model-call budget visible.

By the end, you should be able to:

- Connect the shared ADK `AgentLoop` to ordinary Python retrieval tools.
- Explain how the application restricts both search and passage fetching.
- Read actual tool traces and distinguish completion, refusal, and budget stops.
- Edit the starter factory and plan a comparison against the single-pass baseline.

## 1. Experiment Overview

This notebook builds on [notebook 01](01_harness_and_baseline.ipynb).
Run the cells in order. Leave
`RUN_EXPENSIVE = False` for the first pass. This path reads the measured shared
cache and runs small local exercises; it needs no API key, downloads no model
weights, and makes no hosted calls. The output identifies the original measurement
separately from today's replay. A missing report is an error, not a substitute result.

Enabling the flag loads local BGE embeddings and the optional ADK runtime,
then runs **four questions from two ContractNLI contracts**. Each question has
a six-model-call cap: at most 24 model calls across these examples. The model is
the generation model selected by `choose_models()`, with reasoning disabled and
shared onboarding or `.env` credentials. The saved Qwen run used 15 calls. A fresh run can follow a different path;
temperature zero removes sampling variance but does not guarantee portable outputs.

In [1]:
import sys
from pathlib import Path

from IPython.display import Markdown
from IPython.display import display as render_display


# Replay is the first-run path. Fresh measurements require an explicit opt-in.
RUN_EXPENSIVE = False

REPO = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "aieng-advanced-rag").is_dir()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from within the advanced-RAG checkout.")
RI = REPO / "implementations" / "regulatory_policy_qa"
sys.path.insert(0, str(REPO / "aieng-advanced-rag"))
sys.path.insert(0, str(RI))
from advanced_lab.common import saved  # noqa: E402 - use this source checkout


def show_table(headers, rows):
    """Display a compact table without requiring pandas or a model runtime."""

    def line(values):
        return "| " + " | ".join(str(value).replace("|", r"\|").replace("\n", " ") for value in values) + " |"

    render_display(Markdown("\n".join([line(headers), line(["---"] * len(headers)), *(line(row) for row in rows)])))


print("Execution mode:", "fresh experiment enabled" if RUN_EXPENSIVE else "saved reports + local exercises")

FOLDER = None
QUESTIONS = []  # [{"doc_id": "policy.md", "question": "What notice is required?"}]

Execution mode: saved reports + local exercises


## 2. Understand the Shared Starter Pattern

| Component | Responsibility | Where to inspect it |
| --- | --- | --- |
| Application | Assign the contract and create its isolated index | `advanced_lab/agent.py` |
| `retrieval_tools` | Expose search and, when supported, fetch as typed Python functions | Shared `agents/tools.py` |
| `AgentLoop` | Run model/tool turns and enforce the model-call limit | Shared `agents/loop.py` |
| `audited_tools` | Record tool arguments and returned chunk IDs | `advanced_lab/agent.py` |
| Usage meter | Attribute calls and tokens to each question | Shared `metering` package |

The model chooses when to call a tool, but it does not choose which contract
it may search. `scope` is bound outside the tool's public arguments. The
runner also builds a **physically separate index per contract**, so a guessed
foreign chunk ID cannot widen a fetch. Restricting search alone would not
protect a tool that could fetch arbitrary IDs from a shared index.

This uses the repository's current shared `AgentLoop` and retrieval-tool
pattern. It does not claim conformance to an unavailable future starter template.
The four examples are selected before execution: one answerable and one null
question from each of the first two contracts. They demonstrate wiring and
behavior, not an agent accuracy benchmark.

### Call the tool boundary before creating an agent

These invented passages use model-free BM25, so you can change the question and
inspect actual `retrieval_tools` results without ADK, embeddings, or a model.
Only contract A is indexed. The foreign ID fetch demonstrates that both search
and fetch are bounded by the application. This is a tool exercise, separate from
the saved dense/ADK measurements below.

In [2]:
from aieng.advanced_rag.agents.tools import retrieval_tools  # noqa: E402 - method exercise cell
from aieng.advanced_rag.retrieval.corpus import Chunk  # noqa: E402 - method exercise cell
from aieng.advanced_rag.retrieval.sparse import BM25Retriever  # noqa: E402 - method exercise cell


text = "The receiving party must delete confidential copies within thirty days."
local_retriever = BM25Retriever()
local_retriever.index([Chunk("contract-a::0", "contract-a", text, 0, len(text))])
tools = {tool.__name__: tool for tool in retrieval_tools(local_retriever, k=3, scope="contract-a")}
question = "When must confidential copies be deleted?"
print("Search:", tools["search_corpus"](question))
print("Foreign fetch:", tools["fetch_passages"](["contract-b::0"]))

Search: {'passages': [{'text': 'The receiving party must delete confidential copies within thirty days.', 'citation': 'contract-a', 'chunk_id': 'contract-a::0', 'score': 0.24828292443045505}]}
Foreign fetch: {'passages': []}


## 3. Inspect and Customize the Agent Factory

The factory below is passed to `measure_agent` when the flag is enabled.
The runner supplies the scoped retriever, contract ID, per-question meter,
and trace list. `audited_tools` returns plain callables with typed signatures
and docstrings; ADK supplies the runtime wrapper.

Read the instruction and budget before editing. The instruction asks the model
to search, cite, and refuse unsupported answers. An instruction is not proof
that the model complied: inspect the trace and evaluate the final answer.
Scope and call limits are enforced by the application/runtime, rather than
entrusted to that instruction. Cached mode creates no agent or API client.

In [3]:
if RUN_EXPENSIVE:
    from advanced_lab.agent import audited_tools
    from aieng.advanced_rag.agents.loop import AgentLoop
    from aieng.advanced_rag.models import choose_models
    from aieng.advanced_rag.proxy import resolve_api_key, resolve_base_url

    MODELS = choose_models(show=True)

    def make_starter(retriever, scope, question_meter, trace):
        """Build an editable agent on the shared loop and tool interfaces."""
        return AgentLoop(
            audited_tools(retriever, scope, trace),
            model=MODELS.generation.model,
            base_url=resolve_base_url(start=REPO),
            api_key=resolve_api_key(start=REPO),
            instruction=(
                "Search the scoped contract before answering. Cite the returned passage citations. "
                "If the contract does not state the answer, explicitly say so. Do not use outside knowledge."
            ),
            name="regulatory_qa_starter",
            enable_thinking=False,
            max_model_calls=6,
            charge_to=question_meter,
        )
else:
    print("Editable factory shown above; cached mode creates no agent and makes no model calls.")

Editable factory shown above; cached mode creates no agent and makes no model calls.


## 4. Load or Execute the Four Examples

A fresh run calls `measure_agent(factory=make_starter)`, so edits to the factory
above actually affect the agent being measured. Keep the model and call cap
consistent across questions; the runner checks and records their configured
values. Preserve [agent.json](outputs/advanced/agent.json) before refreshing
the standard report path if you want to compare your variant with the control.

In [4]:
if RUN_EXPENSIVE and FOLDER is None:
    from advanced_lab.agent import measure_agent

    report = measure_agent(factory=make_starter)
else:
    report = saved("agent")

print(
    "Mode:",
    "fresh reference-cohort experiment"
    if RUN_EXPENSIVE and FOLDER is None
    else "saved measured report; no fresh model calls",
)

Mode: saved measured report; no fresh model calls


## 5. Read Stopping Behavior Before Reading Answers

| Recorded field | Interpretation |
| --- | --- |
| `stop_reason = completed` | The loop returned; this does not establish correctness. |
| `stop_reason = model_call_limit` | The runtime stopped further model calls; no successful answer is inferred. |
| `empty_final_answer` | The returned response was empty; this is not a validated semantic refusal. |
| `is_null` | Evaluation label, not a tool argument or the model's decision. |
| `tool_trace` | Actual tool calls and source IDs returned, rather than a narration of intended actions. |

A supported answer, an explicit statement that the contract does not answer,
an empty response, and an exhausted call budget require different evaluation.
The runner does not turn a budget exception into a successful null prediction.

In [5]:
show_table(
    ["Question", "Gold null?", "Stop reason", "Model calls", "Tool calls", "Empty final?"],
    [
        (
            row["query_id"],
            row["is_null"],
            row["stop_reason"],
            row["model_calls"],
            len(row["tool_trace"]),
            row.get("empty_final_answer", "no returned answer"),
        )
        for row in report["results"]
    ],
)
print("Configured maximum model calls per question:", report["max_model_calls"])
print("Recorded run usage:", report["usage"]["run"])

| Question | Gold null? | Stop reason | Model calls | Tool calls | Empty final? |
| --- | --- | --- | --- | --- | --- |
| 1::nda-11 | True | completed | 3 | 2 | False |
| 1::nda-16 | False | completed | 3 | 2 | False |
| 2::nda-11 | False | completed | 3 | 2 | False |
| 2::nda-10 | True | model_call_limit | 6 | 6 | no returned answer |

Configured maximum model calls per question: 6
Recorded run usage: {'questions': 4, 'calls': 15, 'prompt_tokens': 38347, 'completion_tokens': 843, 'total_tokens': 39190, 'tool_calls': 12, 'cache_hits': 0, 'empty_responses': 0, 'calls_missing_usage': 0, 'service_seconds': 86.399}


The saved run has **three completions and one six-call budget stop**, using
**15 model calls** overall. Three completions are not three correct answers.
The saved usage belongs to that measured run; today's replay makes no model
calls and provides no evidence that rerunning would follow an identical trace.

## 6. Inspect One Actual Tool Trace and Its Scope

Select a row by changing `example_index`. Read the question ID, returned source
IDs, and final text together. Does the answer cite evidence that was actually
returned? Did a repeated search add information? The source-ID audit checks
application scope, not answer semantics. The printed text comes from the saved
model response, so it must not be treated as a reference answer.

In [6]:
example_index = 0
example = report["results"][example_index]
print("Question:", example["query_id"], "Assigned contract:", example["doc_id"])
print("Stop:", example["stop_reason"])
for turn, call in enumerate(example["tool_trace"], start=1):
    print("Tool call", turn, call["tool"], "Arguments:", call["arguments"])
    print("Returned chunk IDs:", call["returned_chunk_ids"])
print("Saved model answer:", example["answer"])

returned_ids = [
    chunk_id for row in report["results"] for call in row["tool_trace"] for chunk_id in call["returned_chunk_ids"]
]
observed_scope_ok = all(
    chunk_id.startswith(row["doc_id"] + "::")
    for row in report["results"]
    for call in row["tool_trace"]
    for chunk_id in call["returned_chunk_ids"]
)
print("Returned IDs inspected:", len(returned_ids))
print("All observed returned IDs belong to their assigned contract:", observed_scope_ok)

Question: 1::nda-11 Assigned contract: 1
Stop: completed
Tool call 1 search_corpus Arguments: {'question': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information."}
Returned chunk IDs: ['1::0007', '1::0004', '1::0006', '1::0008', '1::0005']
Tool call 2 fetch_passages Arguments: {'chunk_ids': ['1::0007']}
Returned chunk IDs: ['1::0007']
Saved model answer: The provided contract does not explicitly state that the "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information."

However, it does contain a related restriction under section 4.a. "No Use," which states: "The Receiving Party agrees not to use the Confidential Information in any way, or to manufacture or test any product embodying Confidential Information, except for the purpose set forth above." [Citation: 1]
Returned IDs inspected: 49
All observed returned IDs belong to their assigned contract: True


The trace demonstrates what was returned in these examples. The stronger
application boundary comes from the per-contract index and tool binding;
this observed-ID check alone cannot prove that every possible model action
would be safe. Likewise, a citation and a completed loop do not establish
reference correctness or grounding.

## 7. Adapt the starter to your own corpus

The next cell is off with the same expensive flag. `FOLDER` accepts the text,
Markdown, or PDF files supported by notebook 04's loader. Each manual question
must name a document ID from that folder. No gold answerability is inferred for
these user questions: the report records an unknown label rather than counting
an unlabelled question as a null. Both search and fetch still use an isolated
per-document index. Model and call budget come from the editable factory above.

Set a small folder and one or two questions first. The call cap is per question;
the runner accepts at most 20 starter questions. Live variants are written under
the local cache with a run ID and do not overwrite the shipped four examples.

In [7]:
if RUN_EXPENSIVE and FOLDER is not None:
    import byoc
    from advanced_lab.agent import measure_agent
    from aieng.advanced_rag.retrieval.corpus import Query, Slice, chunk_contract

    documents, _ = byoc.read_corpus(FOLDER, limit=20)
    chunks = [chunk for document in documents for chunk in chunk_contract(document.doc_id, document.text)]
    queries = [Query(f"own-{i}", row["doc_id"], "own-corpus", row["question"]) for i, row in enumerate(QUESTIONS)]
    own_corpus = Slice(chunks, queries, {})
    own_report = measure_agent(factory=make_starter, corpus=own_corpus, labels_known=False)
    print(
        "Own-corpus stopping behavior:",
        [(row["query_id"], row["stop_reason"], row["is_null"]) for row in own_report["results"]],
    )
else:
    print("Own-corpus agent execution is off. The local tool exercise above needs no model.")

Own-corpus agent execution is off. The local tool exercise above needs no model.


## 8. Plan a Controlled Agent Experiment

1. First inspect a budget-stopped row. Separate useful evidence gathering
   from repeated tool use, and state what additional step might help.
2. Edit one element in `make_starter`, such as the instruction. Keep model,
   scope, tools, reader settings, and call budget fixed for the initial A/B.
   Merely editing the factory in replay mode does not change saved results.
3. If testing call budgets, change only `max_model_calls`. Record completion,
   semantic refusal, budget stop, correctness, grounding, calls, and tokens
   separately. The budget is per question, not per individual tool call.
4. Compare with the single-pass pipeline on **identical held-out questions**.
   Do not compare these four examples with notebook 01's different four-question
   sample as if they were a matched accuracy study. Include null controls and
   judge answerables with `judge_answer`, as notebook 05 demonstrates.

**Check your understanding:** Why is fixed search scope insufficient when
fetch can read another contract? Why is a budget stop different from an
abstention? What evidence would justify the extra model calls over single-pass RAG?

## 9. Continue Beyond the Starter

RI #1's next step is a measured experiment, rather than a larger tool list.
Use [ANSWERING_EXPERIMENTS.md](ANSWERING_EXPERIMENTS.md) for public-domain
regulatory QA and RAGAS with local embeddings. Those runners illustrate
answer evaluation; their synthetic questions and small cohorts have explicit
limitations. [GUIDE.md](GUIDE.md) maps shared interfaces to source files,
[PIPELINE.md](PIPELINE.md) explains the recommended single-pass path, and
[CORPUS.md](CORPUS.md) records source rights and attribution.

Keep private source passages, prompts, graph indexes, and response caches in
ignored storage. Share derived reports with their configuration, corpus
provenance, and cost so someone else can assess the experiment.